In [2]:
import pandas as pd
import numpy as np
from ScanResult import *
from data_scan import *
from data_clean_manual import *

In [3]:
np.random.seed(42)
rows = 20

# Create sample data with 10 columns.
data = {
    "num1": np.random.randint(0, 100, size=rows),
    "num2": np.random.randn(rows) * 10,
    "cat1": np.random.choice(["apple", "banana", "cherry"], size=rows),
    "cat2": np.random.choice(["red", "green", "blue"], size=rows),
    "num3": np.random.uniform(0, 50, size=rows),
    "num4": np.random.randint(0, 200, size=rows),
    "text": [f"Sample {i}" for i in range(rows)],
    "bool": np.random.choice([True, False], size=rows),
    "num5": np.random.randn(rows) * 5,
    "cat3": np.random.choice(["X", "Y", "Z"], size=rows),
    "cat4": ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't'],
    "empty_col": [np.nan] * 20 ,
    "cat5": np.random.choice(["X"], size=rows),
}
df = pd.DataFrame(data)

for col in ["num2", "num3", "text", "bool"]:
    missing_indices = np.random.choice(df.index, size=3, replace=False)
    df.loc[missing_indices, col] = np.nan

df.loc[10] = df.loc[0]
df.loc[20] = df.loc[0]

df1 = df.copy()
df2 = df.copy()
df2

C:\Users\zhiha\AppData\Local\Temp\ipykernel_25816\2203771725.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[missing_indices, col] = np.nan


,num1,num2,cat1,cat2,num3,num4,text,bool,num5,cat3,cat4,empty_col,cat5
0,51,14.623781,cherry,green,0.781820,34,Sample 0,False,-3.051610,X,a,NaN,X
1,92,15.387150,apple,blue,21.170074,192,Sample 1,False,-1.580829,X,b,NaN,X
2,14,NaN,cherry,blue,19.744076,100,Sample 2,True,-7.412121,Y,c,NaN,X
3,71,6.034412,cherry,green,14.674409,174,NaN,True,-1.144238,X,d,NaN,X
4,60,-2.510440,banana,blue,0.703991,130,NaN,True,4.813206,Y,e,NaN,X
5,20,-1.638671,apple,red,9.942120,0,Sample 5,True,-1.048462,Y,f,NaN,X
6,82,-14.763297,banana,green,35.567098,4,Sample 6,True,-3.870215,Z,g,NaN,X
7,86,14.869810,banana,red,NaN,141,Sample 7,NaN,-1.798891,Y,h,NaN,X
8,74,NaN,banana,red,30.297999,102,Sample 8,True,3.620416,Z,i,NaN,X
9,74,3.555513,banana,green,46.315044,26,Sample 9,False,-1.278823,X,j,NaN,X


In [4]:
all_null_cols = df1.columns[df1.isnull().all()]
df1.drop(columns=all_null_cols, inplace=True)
scan_results = []
scan_results.extend(scan_df_for_duplicates(df1))
scan_results.extend(scan_df_for_missing(df1))
scan_results.extend(scan_df_for_outliers(df1))
scan_results.extend(scan_df_for_categorical(df1))

scan_results

[ScanResult(row=10, col=-1, message='Row 11 is duplicated', action_type=SRActionType.DEFAULT, actions=[ScanResultAction(title='Duplication Remover', description='Remove the duplicated rows', cleaner='clean_df_for_duplicates', cleaner_id=None, activate=True, data=None)]),
 ScanResult(row=20, col=-1, message='Row 21 is duplicated', action_type=SRActionType.DEFAULT, actions=[ScanResultAction(title='Duplication Remover', description='Remove the duplicated rows', cleaner='clean_df_for_duplicates', cleaner_id=None, activate=True, data=None)]),
 ScanResult(row=2, col=num2, message='Missing value in row 3, column 'num2'', action_type=SRActionType.DEFAULT, actions=[ScanResultAction(title='Missing Value Filler', description='Fill with mean value', cleaner='fill_with_mean', cleaner_id=None, activate=True, data=None), ScanResultAction(title='Missing Value Filler', description='Fill with median value', cleaner='fill_with_median', cleaner_id=None, activate=True, data=None), ScanResultAction(title='M

In [5]:
import random

def choose_one_action_per_scan_result(scan_results):
    for sr in scan_results:
        if sr.actions:
            chosen_action = random.choice(sr.actions)
            for action in sr.actions:
                action.activate = False
            chosen_action.activate = True
            sr.actions = [chosen_action]
    return scan_results


chosen_scan_results = choose_one_action_per_scan_result(scan_results)

chosen_scan_results


[ScanResult(row=10, col=-1, message='Row 11 is duplicated', action_type=SRActionType.DEFAULT, actions=[ScanResultAction(title='Duplication Remover', description='Remove the duplicated rows', cleaner='clean_df_for_duplicates', cleaner_id=None, activate=True, data=None)]),
 ScanResult(row=20, col=-1, message='Row 21 is duplicated', action_type=SRActionType.DEFAULT, actions=[ScanResultAction(title='Duplication Remover', description='Remove the duplicated rows', cleaner='clean_df_for_duplicates', cleaner_id=None, activate=True, data=None)]),
 ScanResult(row=2, col=num2, message='Missing value in row 3, column 'num2'', action_type=SRActionType.DEFAULT, actions=[ScanResultAction(title='Missing Value Filler', description='Delete the row', cleaner='delete_missing_rows', cleaner_id=None, activate=True, data=None)]),
 ScanResult(row=3, col=text, message='Missing value in row 4, column 'text'', action_type=SRActionType.DEFAULT, actions=[ScanResultAction(title='Missing Value Filler', description='

In [6]:
for scan_result in scan_results:
    if scan_result.action_type == SRActionType.DEFAULT:
        if "duplicated" in scan_result.message:
            df1 = clean_df_for_duplicates(scan_result, df1)
        elif "Missing value" in scan_result.message:
            df1 = clean_df_for_missing(scan_result, df1)
        elif "Outlier detected" in scan_result.message:
            df1 = clean_df_for_outlier(scan_result, df1)
        elif "Categorical feature" in scan_result.message:
            df1 = clean_df_for_categorical(scan_result, df1)

df1


Row 8 not found in DataFrame.
Row 15 not found in DataFrame.
Row 17 not found in DataFrame.


d:\SEGP-G6\notebooks\data_clean_manual.py:61: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[scan_result.col].fillna(method='bfill', inplace=True)
d:\SEGP-G6\notebooks\data_clean_manual.py:61: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[scan_result.col].fillna(method='bfill', inplace=True)
d:\SEGP-G6\notebooks\data_clean_manual.py:59: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assig

,num1,num2,num3,num4,num5,cat1_apple,cat1_banana,cat1_cherry,cat2_blue,cat2_green,...,cat4_d,cat4_f,cat4_g,cat4_j,cat4_l,cat4_m,cat4_n,cat4_q,cat4_s,cat4_t
0,51,14.623781,0.781820,34,-3.051610,False,False,True,False,True,...,False,False,False,False,False,False,False,False,False,False
1,92,15.387150,21.170074,192,-1.580829,True,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
3,71,6.034412,14.674409,174,-1.144238,False,False,True,False,True,...,True,False,False,False,False,False,False,False,False,False
5,20,-1.638671,9.942120,0,-1.048462,True,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
6,82,-14.763297,35.567098,4,-3.870215,False,True,False,False,True,...,False,False,True,False,False,False,False,False,False,False
9,74,3.555513,46.315044,26,-1.278823,False,True,False,False,True,...,False,False,False,True,False,False,False,False,False,False
11,99,8.324619,45.747984,14,-6.556621,False,True,False,False,False,...,False,False,False,False,True,False,False,False,False,False
12,23,-2.933991,45.747984,89,-4.351525,False,True,False,False,True,...,False,False,False,False,False,True,False,False,False,False
13,2,-0.298386,22.472534,41,-2.533216,True,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
16,1,-1.402185,4.770506,62,-5.481329,False,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False


In [7]:
from data_clean_auto import *
df2

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
df2 = clean_data(df2, handle_dates=True, generate_report=True)
df2